# Taller: Importación, combinación y exportación de datos
**GEIH 2026 (enero - junio)**

Juan Manuel Vellaizan
María Paula Orozco

## 1. Importar librerías

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings("ignore")


## 2. Mejorar visualización de los dataframes

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 3. Conectarse a Google Drive y establecer la ruta de los datasets

In [11]:
from google.colab import drive
drive.mount('/content/drive')

# OJO: revisa que no haya espacios extra al final del nombre de la carpeta
RUTA_BASE = "/content/drive/MyDrive/IA "   # <-- ajusta al nombre exacto de tu carpeta, incluyendo el espacio

# Se asume que las carpetas de los meses están directamente dentro de RUTA_BASE
RUTA_GEIH = RUTA_BASE

print("Contenido real de RUTA_GEIH (usa esto para confirmar nombres exactos):")
for nombre in os.listdir(RUTA_GEIH):
    print(repr(nombre))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contenido real de RUTA_GEIH (usa esto para confirmar nombres exactos):
'Mayo'
'Junio'
'Abril '
'Enero'
'Febrero'
'Marzo '


In [10]:
print('Contenido de /content/drive/MyDrive:')
for item in os.listdir('/content/drive/MyDrive/IA '):
    print(repr(item))

Contenido de /content/drive/MyDrive:
'Mayo'
'Junio'
'Abril '
'Enero'
'Febrero'
'Marzo '


In [8]:
# Nombres de los meses tal como quieres referirte a ellos en el código
meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio"]

# Mapeo automático: busca, sin importar mayúsculas/minúsculas ni espacios sobrantes,
# la carpeta real que corresponde a cada mes
carpetas_reales = os.listdir(RUTA_GEIH)
mapa_carpetas = {}

for mes in meses:
    match = [c for c in carpetas_reales if c.strip().lower() == mes.lower()]
    if match:
        mapa_carpetas[mes] = match[0]
    else:
        print(f"⚠️  No se encontró carpeta para '{mes}'. Carpetas disponibles: {carpetas_reales}")

print(mapa_carpetas)


{'enero': 'Enero', 'febrero': 'Febrero', 'marzo': 'Marzo ', 'abril': 'Abril ', 'mayo': 'Mayo', 'junio': 'Junio'}


## 4. Cargar los datos en el notebook
Los 4 módulos que trae la GEIH por mes suelen llamarse (verifica el nombre exacto en tu carpeta, puede variar levemente entre 'Caracteristicas generales...' , 'Vivienda y Hogares', 'Fuerza de trabajo' y 'Ocupados'):

In [14]:
# Diccionario para guardar, por mes, los 4 dataframes crudos
datos_mes = {}

# Ajusta estos patrones si el nombre real del archivo difiere
# He actualizado los patrones para ser más robustos frente a caracteres especiales, variaciones en los nombres y la capitalización de la extensión.
patrones_modulos = {
    "generales": "*[Cc]aracteri*general*.CSV", # Matches 'Características generales...' or 'Capítulos de características generales...' with capitalized .CSV
    "hogar": "*hogar*vivienda*.CSV",      # Matches 'Datos del hogar y la\xa0vivienda.CSV'
    "fuerza_trabajo": "*Fuerza*trabajo*.CSV", # Matches 'Fuerza de\xa0trabajo.CSV'
    "ocupados": "*Ocupados*.CSV",         # Matches 'Ocupados.CSV'
}

for mes in meses:
    if mes not in mapa_carpetas:
        continue
    ruta_mes = os.path.join(RUTA_GEIH, mapa_carpetas[mes])
    datos_mes[mes] = {}

    # Primero muestra qué archivos hay realmente en la carpeta del mes
    print(f"\n--- {mes} ({mapa_carpetas[mes]}) ---")
    archivos_en_carpeta = os.listdir(ruta_mes)
    for a in archivos_en_carpeta:
        print(" ", repr(a))

    for nombre_modulo, patron in patrones_modulos.items():
        archivos = glob.glob(os.path.join(ruta_mes, patron))
        if len(archivos) == 0:
            print(f"⚠️  No se encontró archivo para el módulo '{nombre_modulo}' en {mes}. "
                  f"Revisa los nombres listados arriba y ajusta el patrón si es necesario.")
            continue
        # Si hay más de un match, toma el primero y revisa manualmente
        archivo = archivos[0]
        df = pd.read_csv(archivo, encoding="latin-1", sep=None, engine="python") # Removed low_memory=False
        datos_mes[mes][nombre_modulo] = df
        print(f"{mes} - {nombre_modulo}: {df.shape} <- {os.path.basename(archivo)}")


--- enero (Enero) ---
  'Características generales, seguridad social en salud y educación.CSV'
  'Fuerza de\xa0trabajo.CSV'
  'Datos del hogar y la\xa0vivienda.CSV'
  'Ocupados.CSV'
enero - generales: (66606, 55) <- Características generales, seguridad social en salud y educación.CSV
enero - hogar: (24075, 49) <- Datos del hogar y la vivienda.CSV
enero - fuerza_trabajo: (53384, 42) <- Fuerza de trabajo.CSV
enero - ocupados: (28359, 208) <- Ocupados.CSV

--- febrero (Febrero) ---
  'Datos del hogar y la\xa0vivienda (1).CSV'
  'Fuerza de\xa0trabajo (1).CSV'
  'Características generales, seguridad social en salud y educación (1).CSV'
  'Ocupados (1).CSV'
febrero - generales: (67736, 55) <- Características generales, seguridad social en salud y educación (1).CSV
febrero - hogar: (24530, 49) <- Datos del hogar y la vivienda (1).CSV
febrero - fuerza_trabajo: (54234, 42) <- Fuerza de trabajo (1).CSV
febrero - ocupados: (29892, 208) <- Ocupados (1).CSV

--- marzo (Marzo ) ---
  'Fuerz

## 5. Para cada mes, unir los 4 módulos por DIRECTORIO, SECUENCIA_P y ORDEN

In [15]:
claves = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]

def unir_modulos_mes(dic_modulos, claves=claves):
    generales = dic_modulos["generales"]
    hogar = dic_modulos["hogar"]
    fuerza = dic_modulos["fuerza_trabajo"]
    ocupados = dic_modulos["ocupados"]

    # 'hogar' normalmente viene a nivel de vivienda/hogar (sin ORDEN de persona),
    # así que se une solo por DIRECTORIO y SECUENCIA_P si ORDEN no existe en ese módulo
    claves_hogar = [c for c in claves if c in hogar.columns]

    base = generales.merge(hogar, on=claves_hogar, how="left", suffixes=("", "_hogar"))
    base = base.merge(fuerza, on=claves, how="left", suffixes=("", "_fuerza"))
    base = base.merge(ocupados, on=claves, how="left", suffixes=("", "_ocupados"))
    return base

bases_mensuales = {}
for mes in meses:
    if len(datos_mes[mes]) == 4:
        bases_mensuales[mes] = unir_modulos_mes(datos_mes[mes])
        print(f"{mes}: base combinada -> {bases_mensuales[mes].shape}")
    else:
        print(f"⚠️  {mes} no tiene los 4 módulos cargados, revisa los archivos")


enero: base combinada -> (66606, 346)
febrero: base combinada -> (67736, 346)
marzo: base combinada -> (66909, 346)
abril: base combinada -> (67349, 346)
mayo: base combinada -> (67065, 346)
junio: base combinada -> (66285, 346)


## 6. Concatenar las bases de todos los meses (base completa 2026)

In [16]:
geih_2026 = pd.concat(
    [bases_mensuales[mes].assign(MES=mes) for mes in meses if mes in bases_mensuales],
    ignore_index=True
)
print(geih_2026.shape)
geih_2026.head()


(401950, 346)


,PERIODO,MES,PER,DIRECTORIO,SECUENCIA_P,ORDEN,HOGAR,REGIS,AREA,CLASE,FEX_C18,DPTO,PT,P6016,P3271,P6040,P6030S1,P6030S3,P6050,P6083,P6083S1,P6081,P6081S1,P2057,P2059,P2061,P6080,P6080S1,P6080S1A1,P6070,P6071,P6071S1,P6090,P6100,P6110,P6120,P1906S1,P1906S2,P1906S3,P1906S4,P1906S5,P1906S6,P1906S7,P1906S8,P6160,P6170,P3041,P3042,P3042S1,P3042S2,P3043,P3043S1,P3038,P3039,POB_MAY18,PERIODO_hogar,MES_hogar,PER_hogar,HOGAR_hogar,REGIS_hogar,AREA_hogar,CLASE_hogar,FEX_C18_hogar,DPTO_hogar,P4005,P4010,P4020,P4030S1,P4030S1A1,P4030S2,P4030S3,P4030S4,P4030S4A1,P4030S5,P70,P5000,P5010,P5020,P5030,P5040,P5050,P5070,P5080,P5090,P5090S1,P5100,P5110,P5130,P5140,P5222S1,P5222S2,P5222S3,P5222S4,P5222S5,P5222S6,P5222S7,P5222S8,P5222S8A1,P5222S9,P5222S10,P5222S11,P6008,PERIODO_fuerza,MES_fuerza,PER_fuerza,HOGAR_fuerza,REGIS_fuerza,AREA_fuerza,CLASE_fuerza,FEX_C18_fuerza,DPTO_fuerza,FT,FFT,PET,P6240,P6240S1,P6240S2,P6250,P6260,P6260S1,P6260S1A1,P6260S2,P6270,P6280,P3362S1,P3362S2,P3362S3,P3362S4,P3362S5,P3362S6,P3362S7,P3362S8,P3362S7A1,P6300,P6310,P6310S1,P6320,P6330,P6340,P6350,P6351,PERIODO_ocupados,MES_ocupados,PER_ocupados,HOGAR_ocupados,REGIS_ocupados,AREA_ocupados,CLASE_ocupados,FEX_C18_ocupados,DPTO_ocupados,FT_ocupados,P3044S2,P6440,P6450,P6460,P6460S1,P6400,P6410,P6422,P6420S2,P6424S1,P6424S2,P6424S3,P6424S5,P6430,P6430S1,P3045S1,P3045S2,P3045S3,P3046,P3363,P3363S1,P9440,P6500,P3364,P3364S1,P6510,P6510S1,P6510S2,P6590,P6590S1,P6600,P6600S1,P6610,P6610S1,P6620,P6620S1,P6585S1,P6585S1A1,P6585S1A2,P6585S2,P6585S2A1,P6585S2A2,P6585S3,P6585S3A1,P6585S3A2,P6585S4,P6585S4A1,P6585S4A2,P6545,P6545S1,P6545S2,P6580,P6580S1,P6580S2,P6630S1,P6630S1A1,P6630S2,P6630S2A1,P6630S3,P6630S3A1,P6630S4,P6630S4A1,P6630S6,P6630S6A1,P6640,P6640S1,P1800,P1800S1,P1801S1,P1801S2,P1801S3,P1802,P3047,P3048,P3049,P6765,P6765S1,P3051,P3052,P3053,P3365,P3365S1,P3054,P3054S1,P3055,P3055S1,P3056,P3057,P6760,P3058S1,P3058S2,P3058S3,P3058S4,P3058S5,P3059,P3061,P3062S1,P3062S2,P3062S3,P3062S4,P3062S5,P3062S6,P3062S7,P3062S8,P3062S9,P3063,P3063S1,P3064,P3064S1,P3065,P3066,P3560,P3561,P3067,P3067S1,P3067S2,P6775,P3068,P3562,P3563,P6750,P3073,P550,P6780,P6780S1,P1879,P1879S1,P1805,P6790,P6800,P6810,P6810S1,P6850,P6830,P6830S1,P3069,P6880,P6880S1,P6915,P6915S1,P6920,P6930,P6940,P6960,P6990,P9450,P7020,P760,P7026,P7028,P7028S1,P1880,P1880S1,P7040,P7045,P3071S3,P3072S2,P7050,P7070,P7075,P7077,P7090,P7100,P7110,P7120,P7130,P7140S1,P7140S2,P7140S3,P7140S4,P7140S5,P7140S6,P7140S7,P7140S8,P7140S9,P7140S9A1,P7150,P7160,P7170S1,P7170S5,P7170S6,P7180,P514,P515,P1881,P1881S1,P1882,P7240,P7240S1,OCI,INGLABO,RAMA2D_R4,RAMA4D_R4,OFICIO_C8,P6426
0,20260105,enero,2026,8473909,1,1,1,10,NaN,2,41.34,8,1,1,2,34,3.00,"1,991.00",1,2,NaN,2,NaN,2,2.00,2,6,NaN,NaN,2.00,1.00,2.00,1,3.00,NaN,NaN,4,4,4,4,4,4,4,4,1.00,2.00,NaN,4.00,3.00,NaN,NaN,NaN,1.00,2.00,1.00,20260105,1,2026,1,1,NaN,2,41.34,8,1,1,1,1,0.00,2,2,2,NaN,1,1,2,1,2,1.00,3,1,4,1.00,5,NaN,NaN,NaN,"300,000.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,5,"20,260,105.00",1.00,"2,026.00",1.00,50.00,NaN,2.00,41.34,8.00,1.00,NaN,1.00,4.00,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,260,105.00",1.00,"2,026.00",1.00,60.00,NaN,2.00,41.34,8.00,1.00,"4,792.00",2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN,NaN,NaN,3.00,1.00,1.00,1.00,6.00,NaN,NaN,"400,000.00",1.00,2.00,NaN,2.00,NaN,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,2.00,2.00,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"100,000.00",1.00,"1,200,000.00",3.00,NaN,3.00,NaN,2.00,12.00,28.00,2.00,NaN,28.00,NaN,NaN,1.00,5.00,NaN,2.00,NaN,2.00,NaN,NaN,NaN,2.00,2.00,1.00,0.00,24.00,4.00,NaN,9.00,NaN,2.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN,NaN,2.00,NaN,

## 7. Agregar descripción CIIU a 4 dígitos (columna 'RAMA4D_R4')

In [18]:
ruta_ciiu = os.path.join(RUTA_BASE, "2025-01-14 Correlativa CIIU Rev4.xlsx")
ciiu = pd.read_excel(ruta_ciiu, sheet_name="CIIU 2022")

# Revisa los nombres reales de columnas antes de continuar
print(ciiu.columns.tolist())
ciiu.head()


['RAMA4D_R4', 'DESCRIPCION_CIIU']


,RAMA4D_R4,DESCRIPCION_CIIU
0,0,Desconocido
1,10,Actividad Personas Naturales
2,20,Actividad Personas Naturales
3,81,Actividad Personas Naturales
4,82,Actividad Personas Naturales


In [20]:
# Ajusta 'COL_CODIGO_CIIU' y 'COL_DESCRIPCION_CIIU' a los nombres reales de las columnas
# que viste en el paso anterior (ej. 'Código' y 'Descripción')
COL_CODIGO_CIIU = "RAMA4D_R4"
COL_DESCRIPCION_CIIU = "DESCRIPCION_CIIU"

ciiu_slim = ciiu[[COL_CODIGO_CIIU, COL_DESCRIPCION_CIIU]].drop_duplicates()
ciiu_slim = ciiu_slim.rename(columns={
    COL_CODIGO_CIIU: "RAMA4D_R4",
    COL_DESCRIPCION_CIIU: "DESCRIPCION_CIIU"
})

# Asegurar mismo tipo de dato para el merge
geih_2026["RAMA4D_R4"] = geih_2026["RAMA4D_R4"].astype(str).str.strip()
ciiu_slim["RAMA4D_R4"] = ciiu_slim["RAMA4D_R4"].astype(str).str.strip()

geih_2026 = geih_2026.merge(ciiu_slim, on="RAMA4D_R4", how="left")

# Si el merge trajo columnas adicionales por error, bórralas aquí
# geih_2026 = geih_2026.drop(columns=["columna_extra"])

print(geih_2026.shape)
geih_2026[["RAMA4D_R4", "DESCRIPCION_CIIU"]].head()

(401950, 347)


,RAMA4D_R4,DESCRIPCION_CIIU
0,4792.0,NaN
1,9522.0,NaN
2,nan,NaN
3,nan,NaN
4,nan,NaN


## 8. Agregar nombre del departamento (columna 'DPTO') usando Divipola

In [35]:
ruta_divipola = os.path.join(RUTA_BASE, "2025-01-14 DIVIPOLA.xlsx")
divipola = pd.read_excel(ruta_divipola, sheet_name="Departamentos", header=9)

print(divipola.columns.tolist())
divipola.head()

['Código', 'Nombre']


,Código,Nombre
0,05,ANTIOQUIA
1,08,ATLÁNTICO
2,11,"BOGOTÁ, D.C."
3,13,BOLÍVAR
4,15,BOYACÁ


In [40]:
# Ajusta a los nombres reales de columnas vistos arriba
COL_CODIGO_DPTO = "Código"
COL_NOMBRE_DPTO = "Nombre"

divipola_slim = divipola[[COL_CODIGO_DPTO, COL_NOMBRE_DPTO]].drop_duplicates()

# Limpiar divipola_slim: asegurar que 'Código' sea numérico y de 2 dígitos
divipola_slim[COL_CODIGO_DPTO] = divipola_slim[COL_CODIGO_DPTO].astype(str).str.strip()
divipola_slim = divipola_slim[divipola_slim[COL_CODIGO_DPTO].str.match(r'^\d{2}$', na=False)]

divipola_slim = divipola_slim.rename(columns={
    COL_CODIGO_DPTO: "DPTO",
    COL_NOMBRE_DPTO: "NOMBRE_DPTO"
})

geih_2026["DPTO"] = geih_2026["DPTO"].astype(str).str.strip().str.zfill(2) # Rellenar con ceros a la izquierda
divipola_slim["DPTO"] = divipola_slim["DPTO"].astype(str).str.strip()

geih_2026 = geih_2026.merge(divipola_slim, on="DPTO", how="left")

# Si el merge trajo columnas adicionales por error, bórralas aquí
# geih_2026 = geih_2026.drop(columns=["columna_extra"])

print(geih_2026.shape)
geih_2026[["DPTO", "NOMBRE_DPTO"]].head()

(401950, 350)


,DPTO,NOMBRE_DPTO
0,08,ATLÁNTICO
1,08,ATLÁNTICO
2,08,ATLÁNTICO
3,08,ATLÁNTICO
4,08,ATLÁNTICO


## 9. Guardar la base completa GEIH 2026 en CSV

In [41]:
ruta_salida = os.path.join(RUTA_BASE, "GEIH_2026_completa.csv")
geih_2026.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
print("Guardado en:", ruta_salida)


Guardado en: /content/drive/MyDrive/IA /GEIH_2026_completa.csv
